# 01 - Decision Layer Cookbook

The core Metatate flow over the typed-answer contract:

1. discover governed assets
2. get an asset's decision context and business context
3. inspect column meaning, classification, and masking facts
4. read the full rulebook for an asset
5. authorize a proposed use (allow AND deny)
6. validate generated SQL before execution
7. explain a decision by chaining its `decision_id`


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## 1. Discover governed assets


In [ ]:
discovery = client.discover_context()
pd.DataFrame(
    [
        {
            "table": entry["ref"]["table"],
            "column": entry["ref"].get("column"),
            "instructions": entry["instruction_count"],
        }
        for entry in discovery["assets"]
    ]
)


## 2. Decision context for a table

Ranked, cited instructions (the winner first) plus the published business
context. Every row names its policy, scenario, and decision.


In [ ]:
context = client.get_decision_context(asset("customers"))
print(f"state: {context['state']}  effective: {context['effective_decision']}")
print(json.dumps(context["business_context"], indent=2))
pd.DataFrame(
    [
        {
            "scenario": d["scenario_key"],
            "decision": d["decision"],
            "policy": d["provenance"]["policy_name"],
            "family": d["instruction_family"],
        }
        for d in context["decisions"]
    ]
)


## 3. Column meaning facts (classification, PII, masking)


In [ ]:
facts = client.inspect_data_meaning(asset("customers", "email"))
print(json.dumps(facts, indent=2))


## 4. Read the rulebook before you act

The tools above answer one question at a time. `inspect_governance_rules`
returns everything the CURRENT publication has to say about an asset in a
single call — every active rule with its family, scenario, decision,
enforcement mode, and provenance. Agents use it as a planning input:
read the rulebook first, then plan work that will not be blocked later.


In [ ]:
rules = client.inspect_governance_rules(asset("customers"))
print(f"state: {rules['state']}  active rules: {len(rules['rules'])}")
pd.DataFrame(
    [
        {
            "family": rule["instruction_family"],
            "scenario": rule["scenario_key"],
            "decision": rule["decision"],
            "category": rule["category"],
            "policy": rule["provenance"]["policy_name"],
        }
        for rule in rules["rules"]
    ]
)


The rulebook carries the full destination-aware transfer matrix, so an
agent can see that a Salesforce export will be conditional and an
ads-platform export denied BEFORE attempting either (notebook 03
exercises those decisions).


In [ ]:
transfer = next(
    rule
    for rule in rules["rules"]
    if rule["instruction_family"] == "transfer_governance"
)
print(f"transfer decision: {transfer['decision']} ({transfer['scenario_key']})")
print(json.dumps(transfer["parameters"], indent=2))


## 5. Authorize a proposed use

Analytics is a permitted use — Metatate can also just say yes.
Marketing is prohibited: same asset, different scenario, typed deny.


In [ ]:
analytics = client.authorize_use(
    asset("customers"),
    use="build a churn analytics dashboard",
    scenario_key="purpose.allowed_use",
)
print_answer(analytics)


In [ ]:
marketing = client.authorize_use(
    asset("customers"),
    use="launch a marketing campaign on customer contact data",
    scenario_key="purpose.prohibited_use",
)
print_answer(marketing)


## 6. Validate SQL before execution (intent- and column-aware)


In [ ]:
safe = client.validate_query_context(
    "SELECT region, SUM(arr) FROM customers GROUP BY region",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"aggregate query -> {safe['verdict']}")

detail = client.validate_query_context(
    "SELECT customer_name, email FROM customers WHERE region = 'EU'",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"detail query    -> {detail['verdict']} (a masked column is referenced)")
for finding in detail["findings"]:
    for instruction in finding["instructions"]:
        print(f"  {instruction['decision']}: {instruction['decision_reason']}")


## 7. Explain the decision

Every authorize answer carries the `decision_id` of the winning serving row.
`explain_why` resolves it server-side and tells you whether that row is still
in the CURRENT publication.


In [ ]:
explanation = client.explain_why(analytics["decision_id"])
print(f"current: {explanation['current']}")
print(explanation["explanation"])
print(json.dumps(explanation["record"]["provenance"], indent=2))
